# Overview of DVC

This notebook explains DVC as a stage-planning and dvc layer in Deckard: how canonical stages are expanded, grouped, and mapped to `deps`, `outs`, `metrics`, `plots`, and `params`.

Role in the docs flow:
- Read this notebook first for stage topology and dvc semantics.


## Cross-References

For command semantics and edge-case flags, see the official DVC docs:
- [DVC Command Reference](https://dvc.org/doc/command-reference)
- [`dvc repro` documentation](https://dvc.org/doc/command-reference/repro)
- [`dvc push` documentation](https://dvc.org/doc/command-reference/push)
- [`dvc pull` documentation](https://dvc.org/doc/command-reference/pull)
- [Pipelines and Stages](https://dvc.org/doc/user-guide/pipelines)
- [Metrics, Plots, and Params](https://dvc.org/doc/user-guide/experiment-management)

## Deps

DVC dependencies (`deps`) declare source inputs and code paths that invalidate a stage when changed.

From `docs/notebooks/dvc.yaml`, the `notebook_dvc` stage currently defines:

```yaml
notebook_dvc:
  deps:
    - dvc.ipynb
    - ../../deckard/experiment/
    - ../../deckard/layers/
    - ../../deckard/file.py
    - ../../deckard/utils.py
    - ../../examples/sklearn/config/default.yaml
    - ../../examples/sklearn/config/files/default.yaml
    - ../../examples/sklearn/config/attack/hsj.yaml
    - ../../examples/sklearn/config/defense/class-labels.yaml
    - ../../examples/sklearn/config/plot/
```

## Outs

DVC outputs (`outs`) declare materialized artifacts tracked by the stage.

`notebook_dvc` is intentionally inspection-only and does not persist outs.

A concrete `outs` example from `docs/notebooks/dvc.yaml` (`notebook_optuna`):

```yaml
notebook_optuna:
  outs:
    - ./build/notebook_artifacts/optuna/single_study.pkl
    - ./build/notebook_artifacts/optuna/multi_study.pkl
    - ./build/notebook_artifacts/optuna/single_best_params.json
    - ./build/notebook_artifacts/optuna/multi_best_params.json
```

## Metrics

DVC metrics (`metrics`) are structured, diff-friendly values used for evaluation and CI comparison.

`notebook_dvc` has no metrics by design. Metrics are demonstrated in runtime notebooks.

Concrete metrics example from `docs/notebooks/dvc.yaml` (`notebook_dvclive`):

```yaml
notebook_dvclive:
  metrics:
    - ./build/dvclive/dvclive/summary.json
```

## Plots

DVC plots (`plots`) are visualization-oriented artifacts for trend inspection and report rendering.

`notebook_dvc` has no plots by design. Plot persistence is demonstrated in runtime notebooks.

Concrete plots example from `docs/notebooks/dvc.yaml` (`notebook_dvclive`):

```yaml
notebook_dvclive:
  plots:
    - ./build/dvclive/dvclive_feature_spec.vl.json
```

## Params

DVC params (`params`) capture configuration inputs that define run identity and reproducibility.

For this notebook pipeline file (`docs/notebooks/dvc.yaml`), no stage currently declares a top-level `params:` key.

In this setup, reproducibility-relevant inputs are primarily encoded through `deps` on config files (for example `../../examples/sklearn/config/default.yaml`).

If you want explicit params tracking here, add a `params:` block to a stage and point it at specific config keys/files.

## dvc repro

Use this command to (re)run pipeline stages whose dependencies or params changed.

From `docs/notebooks/dvc.yaml`, this notebook is wired as stage `notebook_dvc`:

```yaml
notebook_dvc:
  cmd: jupyter nbconvert --to notebook --execute --inplace dvc.ipynb
```

Run only this stage:

```bash
dvc repro notebook_dvc
```

Force-run notebook-prefixed stages (useful after notebook refactors):

```bash
dvc repro --force notebook_*
```

What it does:
- Recomputes stage state from declared dependencies.
- Executes stage command when inputs changed (or when forced).
- Refreshes tracked artifacts for stages that declare `outs`/`metrics`/`plots`.

## dvc push

Use this command to upload cache objects from local cache to configured remote storage.

```bash
dvc push
```

In this notebook pipeline file, `notebook_dvc` itself does not declare `outs`/`metrics`/`plots`, but sibling stages do (for example `notebook_dvclive`, `notebook_optuna`, `notebook_lifelines`).

Typical sequence after reproducing artifact-producing stages:

```bash
dvc repro notebook_dvclive notebook_optuna && dvc push
```

What it does:
- Transfers cache objects for tracked stage artifacts to remote.
- Enables collaborators/CI to retrieve identical artifacts via `dvc pull`.

## dvc pull

Use this command to download required cache objects from remote into local cache and workspace.

```bash
dvc pull
```

Use after updating Git metadata (for example, after pulling a commit that changes `docs/notebooks/dvc.yaml` or lockfiles):

```bash
git pull && dvc pull
```

What it does:
- Restores tracked outputs for stages that materialize artifacts.
- Aligns local workspace artifacts with remote-backed DVC state.
- Keeps local runs consistent with collaborator and CI outputs.

This notebook intentionally excludes DVCLive runtime artifact demonstrations, which are covered in `dvclive.ipynb`.